In [ ]:
"""
Sales Deal Win Probability Model — Training Skeleton
------------------------------------------------------
Expects a CLEANED csv from the data-prep teammate with (at minimum) these columns:

    opportunity_number
    opportunity_result              <- target, values like "Won"/"Lost"
    opportunity_amount_usd
    elapsed_days_in_sales_stage
    sales_stage_change_count
    total_days_identified_through_closing
    client_size_by_revenue
    route_to_market                 <- categorical
    supplies_subgroup                <- categorical
    competitor_type                  <- categorical
    revenue_from_client_past_two_years

If your teammate's cleaned column names differ, just update CATEGORICAL_COLS / NUMERIC_COLS below.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import lightgbm as lgb
import joblib

# ---------------------------------------------------------------------------
# 1. CONFIG — adjust these to match the cleaned dataset's real column names
# ---------------------------------------------------------------------------

DATA_PATH = "cleaned_deals.csv"        # <- swap for wherever the teammate saves their output
TARGET_COL = "opportunity_result"       # win/loss column
POSITIVE_LABEL = "Won"                  # whatever string means "won" in the cleaned data

CATEGORICAL_COLS = [
    "route_to_market",
    "supplies_subgroup",
    "competitor_type",
]

NUMERIC_COLS = [
    "elapsed_days_in_sales_stage",
    "sales_stage_change_count",
    "total_days_identified_through_closing",
    "client_size_by_revenue",
    "revenue_from_client_past_two_years",
]

FEATURES = CATEGORICAL_COLS + NUMERIC_COLS

MODEL_OUT_PATH = "win_probability_model.pkl"


In [ ]:
# ---------------------------------------------------------------------------
# 2. LOAD + BASIC CHECKS
# ---------------------------------------------------------------------------

def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    missing_cols = [c for c in FEATURES + [TARGET_COL] if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"Expected columns missing from cleaned data: {missing_cols}. "
            "Update CATEGORICAL_COLS/NUMERIC_COLS/TARGET_COL to match the real schema."
        )

    return df

In [ ]:
# ---------------------------------------------------------------------------
# 3. PREP FOR LIGHTGBM (native categorical support — cast, don't one-hot)
# ---------------------------------------------------------------------------

def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CATEGORICAL_COLS:
        df[col] = df[col].astype("category")
    return df

In [ ]:
# ---------------------------------------------------------------------------
# 4. TRAIN
# ---------------------------------------------------------------------------

def train_model(df: pd.DataFrame):
    y = (df[TARGET_COL] == POSITIVE_LABEL).astype(int)
    X = df[FEATURES]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=CATEGORICAL_COLS)
    val_data = lgb.Dataset(X_test, label=y_test, categorical_feature=CATEGORICAL_COLS, reference=train_data)

    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "seed": 42,
    }

    model = lgb.train(
        params,
        train_data,
        num_boost_round=300,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=50)],
    )

    return model, X_test, y_test

In [ ]:
# ---------------------------------------------------------------------------
# 5. EVALUATE
# ---------------------------------------------------------------------------

def evaluate(model, X_test, y_test):
    y_prob = model.predict(X_test, num_iteration=model.best_iteration)
    y_pred = (y_prob >= 0.5).astype(int)

    print("AUC:", roc_auc_score(y_test, y_prob))
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    # Feature importance — useful for the demo ("here's what drives the prediction")
    importance = pd.Series(
        model.feature_importance(importance_type="gain"), index=model.feature_name()
    ).sort_values(ascending=False)
    print("\nFeature importance:\n", importance)


In [ ]:
# ---------------------------------------------------------------------------
# 6. EXPECTED REVENUE HELPER
#    (not a separate model — win_prob * quoted amount, see project problem statement)
# ---------------------------------------------------------------------------

def add_expected_revenue(df: pd.DataFrame, model) -> pd.DataFrame:
    df = df.copy()
    X = prepare_features(df)[FEATURES]
    df["win_probability"] = model.predict(X, num_iteration=model.best_iteration)
    df["expected_revenue"] = df["win_probability"] * df["opportunity_amount_usd"]
    return df

In [ ]:
# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    raw_df = load_data(DATA_PATH)
    prepped_df = prepare_features(raw_df)

    model, X_test, y_test = train_model(prepped_df)
    evaluate(model, X_test, y_test)

    joblib.dump(model, MODEL_OUT_PATH)
    print(f"\nModel saved to {MODEL_OUT_PATH}")

    # Quick sanity check on a few rows
    scored = add_expected_revenue(raw_df, model)
    print(scored[["opportunity_number", "win_probability", "expected_revenue"]].head())